## Import libraries and load source data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Feature Engineering for Fantasy Football ML Model")
print("="*80)
print("\nThis notebook transforms raw stats into ML-ready features.")
print("Key principle: All historical metrics are SHIFTED by 1 week to prevent data leakage.\n")

Feature Engineering for Fantasy Football ML Model

This notebook transforms raw stats into ML-ready features.
Key principle: All historical metrics are SHIFTED by 1 week to prevent data leakage.



## Install nfl_data_py library

In [2]:
# Install required packages for NFL data extraction
%pip install --no-deps nfl_data_py
%pip install appdirs fastparquet pandas

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 715.8/715.8 kB 7.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fastparquet]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nfl-data-py 0.3.3 requires numpy<2.0,>=1.0, but you have numpy 2.3.5 which is incompatible.
nfl-data-py 0.3.3 requires pandas<2.0,>=1.0, but you have pandas 2.3.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Run nfl_py_extraction notebook to get source data

In [3]:
# Load source data directly using nfl_data_py library
import nfl_data_py as nfl
import pandas as pd

print("Loading data from nfl_data_py library...")

# Load 2025 play-by-play data
pbp_2025 = nfl.import_pbp_data([2025], downcast=True)
print(f"Loaded {len(pbp_2025)} plays from 2025 season")

# Load schedules
schedules = nfl.import_schedules([2025])
print(f"Loaded {len(schedules)} games from schedules")

# Load snap counts (for snap share)
snap_counts = nfl.import_snap_counts([2025])
print(f"Loaded {len(snap_counts)} snap count records")

# Load injury reports (for supporting cast health)
injuries = nfl.import_injuries([2025])
print(f"Loaded {len(injuries)} injury records")

# Load rosters (official positions incl. TE, and PFR->GSIS id mapping)
rosters = nfl.import_seasonal_rosters([2025])
print(f"Loaded {len(rosters)} roster records")

# Calculate weekly stats (from the extraction notebook logic)
print("\nCalculating weekly player statistics from play-by-play data...")

# Passer aggregations
passing_plays = pbp_2025[pbp_2025['play_type'] == 'pass'].copy()
passer_stats = passing_plays.groupby(
    ['week', 'passer_player_id', 'passer_player_name'], 
    dropna=False
).agg({
    'pass_attempt': 'sum',
    'complete_pass': 'sum',
    'yards_gained': 'sum',
    'pass_touchdown': 'sum',
    'interception': 'sum'
}).reset_index()
passer_stats.rename(columns={
    'passer_player_id': 'player_id',
    'passer_player_name': 'player_name',
    'pass_attempt': 'pass_attempts',
    'complete_pass': 'completions',
    'yards_gained': 'passing_yards',
    'pass_touchdown': 'passing_tds',
    'interception': 'interceptions'
}, inplace=True)

# Rusher aggregations
rushing_plays = pbp_2025[pbp_2025['play_type'] == 'run'].copy()
rusher_stats = rushing_plays.groupby(
    ['week', 'rusher_player_id', 'rusher_player_name'],
    dropna=False
).agg({
    'rush_attempt': 'sum',
    'yards_gained': 'sum',
    'rush_touchdown': 'sum'
}).reset_index()
rusher_stats.rename(columns={
    'rusher_player_id': 'player_id',
    'rusher_player_name': 'player_name',
    'rush_attempt': 'rush_attempts',
    'yards_gained': 'rushing_yards',
    'rush_touchdown': 'rushing_tds'
}, inplace=True)

# Receiver aggregations
receiving_plays = pbp_2025[
    (pbp_2025['play_type'] == 'pass') & 
    (pbp_2025['receiver_player_id'].notna())
].copy()
receiver_stats = receiving_plays.groupby(
    ['week', 'receiver_player_id', 'receiver_player_name'],
    dropna=False
).agg({
    'pass_attempt': 'sum',
    'complete_pass': 'sum',
    'yards_gained': 'sum',
    'pass_touchdown': 'sum'
}).reset_index()
receiver_stats.rename(columns={
    'receiver_player_id': 'player_id',
    'receiver_player_name': 'player_name',
    'pass_attempt': 'targets',
    'complete_pass': 'receptions',
    'yards_gained': 'receiving_yards',
    'pass_touchdown': 'receiving_tds'
}, inplace=True)

# Merge all stats
weekly_stats = passer_stats.merge(rusher_stats, on=['week', 'player_id', 'player_name'], how='outer')
weekly_stats = weekly_stats.merge(receiver_stats, on=['week', 'player_id', 'player_name'], how='outer')
weekly_stats['player_name'] = weekly_stats['player_name'].ffill().bfill()

numeric_cols = [
    'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
    'rush_attempts', 'rushing_yards', 'rushing_tds',
    'targets', 'receptions', 'receiving_yards', 'receiving_tds'
]
weekly_stats[numeric_cols] = weekly_stats[numeric_cols].fillna(0)

# Calculate fantasy points (PPR)
weekly_stats['fantasy_points_ppr'] = (
    (weekly_stats['passing_yards'] * 0.04) +
    (weekly_stats['passing_tds'] * 4) +
    (weekly_stats['interceptions'] * -2) +
    (weekly_stats['rushing_yards'] * 0.1) +
    (weekly_stats['rushing_tds'] * 6) +
    (weekly_stats['receiving_yards'] * 0.1) +
    (weekly_stats['receiving_tds'] * 6) +
    (weekly_stats['receptions'] * 1)
)
weekly_stats['fantasy_points_ppr'] = weekly_stats['fantasy_points_ppr'].round(2)
weekly_stats = weekly_stats.sort_values('fantasy_points_ppr', ascending=False).reset_index(drop=True)

print(f"\n✓ Data loaded and ready for feature engineering:")
print(f"  - weekly_stats: {len(weekly_stats):,} player-week records")
print(f"  - schedules: {len(schedules):,} games")
print(f"  - pbp_2025: {len(pbp_2025):,} plays")

Loading data from nfl_data_py library...
2025 done.
Downcasting floats.
Loaded 48771 plays from 2025 season
Loaded 285 games from schedules

Calculating weekly player statistics from play-by-play data...

✓ Data loaded and ready for feature engineering:
  - weekly_stats: 5,613 player-week records
  - schedules: 285 games
  - pbp_2025: 48,771 plays


## Identify player positions and teams from PBP data

In [4]:
# ============================================================================
# STEP 1: Identify player positions and teams from play-by-play data
# ============================================================================
print("Step 1: Identifying player positions and teams...\n")

# Extract position from PBP data (passers = QB, rushers = RB, receivers = WR/TE)
passers = pbp_2025[pbp_2025['passer_player_id'].notna()][['passer_player_id', 'passer_player_name', 'posteam', 'week']].copy()
passers.columns = ['player_id', 'player_name', 'team', 'week']
passers['position'] = 'QB'

rushers = pbp_2025[pbp_2025['rusher_player_id'].notna()][['rusher_player_id', 'rusher_player_name', 'posteam', 'week']].copy()
rushers.columns = ['player_id', 'player_name', 'team', 'week']
rushers['position'] = 'RB'  # Default to RB, we'll refine this

receivers = pbp_2025[pbp_2025['receiver_player_id'].notna()][['receiver_player_id', 'receiver_player_name', 'posteam', 'week']].copy()
receivers.columns = ['player_id', 'player_name', 'team', 'week']
receivers['position'] = 'WR'  # Default to WR, we'll refine based on usage patterns

# Combine all players
all_players = pd.concat([passers, rushers, receivers], ignore_index=True)

# For each player, take the most common position (mode)
player_position = all_players.groupby('player_id').agg({
    'player_name': 'first',
    'position': lambda x: x.mode()[0] if not x.mode().empty else 'FLEX'
}).reset_index()

# Override the heuristic with official roster positions where available.
# This adds TE (the heuristic can't detect it) and fixes pass-catching RBs
# that would otherwise be tagged WR.
roster_positions = rosters[['player_id', 'position']].dropna().drop_duplicates('player_id')
roster_positions.columns = ['player_id', 'roster_position']

player_position = player_position.merge(roster_positions, on='player_id', how='left')
use_roster = player_position['roster_position'].isin(['QB', 'RB', 'WR', 'TE', 'FB'])
player_position.loc[use_roster, 'position'] = (
    player_position.loc[use_roster, 'roster_position'].replace({'FB': 'RB'})
)
player_position.drop(columns=['roster_position'], inplace=True)

print(f"Identified positions for {len(player_position)} unique players:")
print(player_position['position'].value_counts())

# Get most recent team for each player
player_team = all_players.sort_values('week').groupby('player_id').agg({
    'team': 'last'
}).reset_index()
player_team.columns = ['player_id', 'recent_team']

# Merge position and team into weekly_stats
weekly_stats_df = weekly_stats.copy()
weekly_stats_df = weekly_stats_df.merge(player_position, on='player_id', how='left', suffixes=('', '_lookup'))
weekly_stats_df = weekly_stats_df.merge(player_team, on='player_id', how='left')

# Use lookup values to fill missing player_name if needed
weekly_stats_df['player_name'] = weekly_stats_df['player_name'].fillna(weekly_stats_df['player_name_lookup'])
weekly_stats_df.drop(columns=['player_name_lookup'], inplace=True, errors='ignore')

print(f"\nEnriched weekly_stats with position and team data.")
print(f"Shape: {weekly_stats_df.shape}")
display(weekly_stats_df.head())

Step 1: Identifying player positions and teams...

Identified positions for 611 unique players:
position
WR    375
RB    156
QB     80
Name: count, dtype: int64

Enriched weekly_stats with position and team data.
Shape: (5613, 18)


,week,player_id,player_name,pass_attempts,completions,passing_yards,passing_tds,interceptions,rush_attempts,rushing_yards,rushing_tds,targets,receptions,receiving_yards,receiving_tds,fantasy_points_ppr,position,recent_team
0,12,00-0039139,J.Gibbs,0.0,0.0,0.0,0.0,0.0,15.0,219.0,2.0,12.0,11.0,45.0,1.0,55.400002,RB,DET
1,10,00-0036223,J.Taylor,0.0,0.0,0.0,0.0,0.0,32.0,244.0,3.0,4.0,3.0,42.0,0.0,49.599998,RB,IND
2,16,00-0039075,P.Nacua,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.0,12.0,225.0,2.0,46.500000,WR,LA
3,15,00-0036970,K.Pitts,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,11.0,166.0,3.0,45.599998,WR,ATL
4,17,00-0032764,D.Henry,0.0,0.0,0.0,0.0,0.0,36.0,216.0,4.0,0.0,0.0,0.0,0.0,45.599998,RB,BAL


## CATEGORY 1: General Game Context Features

In [5]:
# ============================================================================
# CATEGORY 1: General Game Context (Applies to all players)
# ============================================================================
print("\nStep 2: Engineering General Game Context Features...\n")

# Prepare schedules data
schedules_df = schedules.copy()

# Calculate implied team totals.
# nflverse convention: spread_line is POSITIVE when the HOME team is favored
# (e.g. spread_line = 3.5 means home team favored by 3.5).
schedules_df['home_implied_total'] = (schedules_df['total_line'] / 2) + (schedules_df['spread_line'] / 2)
schedules_df['away_implied_total'] = (schedules_df['total_line'] / 2) - (schedules_df['spread_line'] / 2)

# Weather flags.
# Note: precipitation is NOT available in nfl_data_py schedules (only temp/wind/roof),
# so is_dome captures weather-proof venues instead.
schedules_df['is_dome'] = schedules_df['roof'].isin(['dome', 'closed']).astype(int)
schedules_df['is_bad_weather'] = (
    ((schedules_df['wind'] > 15) | (schedules_df['temp'] < 32)) &
    (schedules_df['is_dome'] == 0)
).astype(int)

# Create home/away context for each team
home_context = schedules_df[['week', 'home_team', 'home_implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']].copy()
home_context.columns = ['week', 'team', 'implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']
home_context['is_home'] = 1
home_context['opponent'] = schedules_df['away_team']
home_context['team_spread'] = schedules_df['spread_line']    # positive = this team favored
home_context['starting_qb_id'] = schedules_df['home_qb_id']

away_context = schedules_df[['week', 'away_team', 'away_implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']].copy()
away_context.columns = ['week', 'team', 'implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']
away_context['is_home'] = 0
away_context['opponent'] = schedules_df['home_team']
away_context['team_spread'] = -schedules_df['spread_line']   # positive = this team favored
away_context['starting_qb_id'] = schedules_df['away_qb_id']

# Combine home and away
game_context = pd.concat([home_context, away_context], ignore_index=True)

# Calculate rest advantage (days since last game)
game_context['gameday'] = pd.to_datetime(game_context['gameday'])
game_context = game_context.sort_values(['team', 'gameday'])
game_context['days_since_last_game'] = game_context.groupby('team')['gameday'].diff().dt.days

# Merge opponent's days since last game to calculate rest advantage
opponent_rest = game_context[['week', 'team', 'days_since_last_game']].copy()
opponent_rest.columns = ['week', 'opponent', 'opp_days_since_last_game']

game_context = game_context.merge(opponent_rest, on=['week', 'opponent'], how='left')
game_context['rest_advantage'] = game_context['days_since_last_game'] - game_context['opp_days_since_last_game']
game_context['rest_advantage'] = game_context['rest_advantage'].fillna(0)

# Merge game context into weekly_stats_df
weekly_stats_df = weekly_stats_df.merge(
    game_context[['week', 'team', 'implied_total', 'team_spread', 'is_home', 'temp', 'wind',
                  'is_bad_weather', 'is_dome', 'rest_advantage', 'opponent', 'starting_qb_id']],
    left_on=['week', 'recent_team'],
    right_on=['week', 'team'],
    how='left'
)

# Drop duplicate team column
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

print("✓ Added game context features:")
print("  - implied_total (Vegas implied team total)")
print("  - team_spread (Vegas spread from team's perspective, positive = favored)")
print("  - is_home (1=home, 0=away)")
print("  - temp, wind (weather conditions)")
print("  - is_bad_weather (wind>15mph or temp<32F, outdoor games only)")
print("  - is_dome (weather-proof venue; precipitation not available in nfl_data_py)")
print("  - rest_advantage (days since last game difference)")
print("  - starting_qb_id (scheduled starting QB, used for QB context features)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")


Step 2: Engineering General Game Context Features...

✓ Added game context features:
  - implied_total (Vegas implied team total)
  - is_home (1=home, 0=away)
  - temp, wind (weather conditions)
  - is_bad_weather (wind>15mph or temp<32F)
  - rest_advantage (days since last game difference)

Current shape: (5613, 25)


## CATEGORY 2: Rolling Averages (All Players)

In [6]:
# ============================================================================
# CATEGORY 2: Rolling Averages (3-Week & 5-Week) - For ALL Players
# ============================================================================
print("\nStep 3: Calculating Rolling Averages (SHIFTED to prevent data leakage)...\n")

# Sort by player and week to ensure proper rolling calculations
weekly_stats_df = weekly_stats_df.sort_values(['player_id', 'week']).reset_index(drop=True)

# Calculate SHIFTED rolling averages for fantasy points
weekly_stats_df['fantasy_points_3wk_avg'] = (
    weekly_stats_df.groupby('player_id')['fantasy_points_ppr']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df['fantasy_points_5wk_avg'] = (
    weekly_stats_df.groupby('player_id')['fantasy_points_ppr']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

print("✓ Added rolling averages for ALL players:")
print("  - fantasy_points_3wk_avg (shifted 3-week average)")
print("  - fantasy_points_5wk_avg (shifted 5-week average)")
print("\n  Note: All rolling metrics use .shift(1) to prevent data leakage.")
print(f"\nCurrent shape: {weekly_stats_df.shape}")


Step 3: Calculating Rolling Averages (SHIFTED to prevent data leakage)...

✓ Added rolling averages for ALL players:
  - fantasy_points_3wk_avg (shifted 3-week average)
  - fantasy_points_5wk_avg (shifted 5-week average)

  Note: All rolling metrics use .shift(1) to prevent data leakage.

Current shape: (5613, 27)


## CATEGORY 3: QB-Specific Features

In [7]:
# ============================================================================
# CATEGORY 3: Quarterback (QB) Specific Features
# ============================================================================
print("\nStep 4: Engineering QB-Specific Features...\n")

# Filter for QBs only
qb_mask = weekly_stats_df['position'] == 'QB'

# Calculate SHIFTED rolling averages for passing attempts and rushing yards
weekly_stats_df.loc[qb_mask, 'qb_pass_attempts_3wk_avg'] = (
    weekly_stats_df[qb_mask].groupby('player_id')['pass_attempts']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df.loc[qb_mask, 'qb_pass_attempts_5wk_avg'] = (
    weekly_stats_df[qb_mask].groupby('player_id')['pass_attempts']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

weekly_stats_df.loc[qb_mask, 'qb_rushing_yards_3wk_avg'] = (
    weekly_stats_df[qb_mask].groupby('player_id')['rushing_yards']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df.loc[qb_mask, 'qb_rushing_yards_5wk_avg'] = (
    weekly_stats_df[qb_mask].groupby('player_id')['rushing_yards']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- Sack rates (pressure context) ---
# Opponent sack rate = how often the opposing defense sacks the QB (matchup).
# Team sack rate allowed = how often this player's own O-line gives up sacks
# (proxy for O-line strength, since O-line rankings are premium data).
# Both are SHIFTED season-to-date rates: cumulative sacks / cumulative dropbacks
# through the previous week.
dropbacks = pbp_2025[pbp_2025['qb_dropback'] == 1]

def shifted_sack_rate(df, team_col, rate_name):
    g = df.groupby([team_col, 'week']).agg(
        dropbacks=('qb_dropback', 'sum'),
        sacks=('sack', 'sum')
    ).reset_index().sort_values('week')
    cum_db = g.groupby(team_col)['dropbacks'].transform(lambda x: x.shift(1).expanding().sum())
    cum_sk = g.groupby(team_col)['sacks'].transform(lambda x: x.shift(1).expanding().sum())
    g[rate_name] = (cum_sk / cum_db).fillna(0)
    return g[[team_col, 'week', rate_name]]

off_sack_rate = shifted_sack_rate(dropbacks, 'posteam', 'team_sack_rate_allowed')
def_sack_rate = shifted_sack_rate(dropbacks, 'defteam', 'opp_def_sack_rate')

weekly_stats_df = weekly_stats_df.merge(
    off_sack_rate, left_on=['recent_team', 'week'], right_on=['posteam', 'week'], how='left'
).drop(columns=['posteam'], errors='ignore')

weekly_stats_df = weekly_stats_df.merge(
    def_sack_rate, left_on=['opponent', 'week'], right_on=['defteam', 'week'], how='left'
).drop(columns=['defteam'], errors='ignore')

weekly_stats_df['team_sack_rate_allowed'] = weekly_stats_df['team_sack_rate_allowed'].fillna(0)
weekly_stats_df['opp_def_sack_rate'] = weekly_stats_df['opp_def_sack_rate'].fillna(0)

qb_count = qb_mask.sum()
print(f"✓ Added QB-specific features for {qb_count} QB-week records:")
print("  - qb_pass_attempts_3wk_avg & 5wk_avg (shifted)")
print("  - qb_rushing_yards_3wk_avg & 5wk_avg (shifted, captures rushing floor)")
print("  - opp_def_sack_rate (shifted season-to-date, matchup pressure)")
print("  - team_sack_rate_allowed (shifted season-to-date, O-line strength proxy)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")


Step 4: Engineering QB-Specific Features...

✓ Added QB-specific features for 664 QB-week records:
  - qb_pass_attempts_3wk_avg & 5wk_avg (shifted)
  - qb_rushing_yards_3wk_avg & 5wk_avg (shifted, captures rushing floor)

Current shape: (5613, 31)


## CATEGORY 4: RB-Specific Features

In [8]:
# ============================================================================
# CATEGORY 4: Running Back (RB) Specific Features
# ============================================================================
print("\nStep 5: Engineering RB-Specific Features...\n")

# --- 4A: Opportunity Share ---
print("  Calculating Opportunity Share...")

# Calculate team totals per week
team_opportunities = pbp_2025.groupby(['posteam', 'week']).agg({
    'rush_attempt': 'sum',
    'pass_attempt': 'sum'  # Targets come from pass attempts
}).reset_index()
team_opportunities['team_total_opportunities'] = (
    team_opportunities['rush_attempt'] + team_opportunities['pass_attempt']
)
team_opportunities = team_opportunities[['posteam', 'week', 'team_total_opportunities']]
team_opportunities.columns = ['team', 'week', 'team_total_opportunities']

# Calculate player opportunities (rush attempts + targets)
weekly_stats_df['player_opportunities'] = (
    weekly_stats_df['rush_attempts'] + weekly_stats_df['targets']
)

# Merge team totals
weekly_stats_df = weekly_stats_df.merge(
    team_opportunities,
    left_on=['recent_team', 'week'],
    right_on=['team', 'week'],
    how='left'
)
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

# Calculate opportunity share
weekly_stats_df['opportunity_share'] = (
    weekly_stats_df['player_opportunities'] / weekly_stats_df['team_total_opportunities']
).fillna(0)

# SHIFTED 3-week and 5-week rolling averages for RBs only
rb_mask = weekly_stats_df['position'] == 'RB'
weekly_stats_df.loc[rb_mask, 'rb_opportunity_share_3wk_avg'] = (
    weekly_stats_df[rb_mask].groupby('player_id')['opportunity_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[rb_mask, 'rb_opportunity_share_5wk_avg'] = (
    weekly_stats_df[rb_mask].groupby('player_id')['opportunity_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 4B: High-Value Touches (HVTs) ---
print("  Calculating High-Value Touches (yardline <= 10)...")

# Count carries and targets inside the 10-yard line
hvt_carries = pbp_2025[
    (pbp_2025['play_type'] == 'run') & 
    (pbp_2025['yardline_100'] <= 10) &
    (pbp_2025['rusher_player_id'].notna())
].groupby(['rusher_player_id', 'week']).size().reset_index(name='hvt_carries')
hvt_carries.columns = ['player_id', 'week', 'hvt_carries']

hvt_targets = pbp_2025[
    (pbp_2025['play_type'] == 'pass') & 
    (pbp_2025['yardline_100'] <= 10) &
    (pbp_2025['receiver_player_id'].notna())
].groupby(['receiver_player_id', 'week']).size().reset_index(name='hvt_targets')
hvt_targets.columns = ['player_id', 'week', 'hvt_targets']

# Merge HVTs into main dataframe
weekly_stats_df = weekly_stats_df.merge(hvt_carries, on=['player_id', 'week'], how='left')
weekly_stats_df = weekly_stats_df.merge(hvt_targets, on=['player_id', 'week'], how='left')
weekly_stats_df['hvt_carries'] = weekly_stats_df['hvt_carries'].fillna(0)
weekly_stats_df['hvt_targets'] = weekly_stats_df['hvt_targets'].fillna(0)
weekly_stats_df['total_hvts'] = weekly_stats_df['hvt_carries'] + weekly_stats_df['hvt_targets']

# SHIFTED 3-week and 5-week rolling averages for RBs only
weekly_stats_df.loc[rb_mask, 'rb_hvts_3wk_avg'] = (
    weekly_stats_df[rb_mask].groupby('player_id')['total_hvts']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[rb_mask, 'rb_hvts_5wk_avg'] = (
    weekly_stats_df[rb_mask].groupby('player_id')['total_hvts']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 4C: Snap Share ---
print("  Calculating Snap Share...")

# Snap counts are keyed by PFR player ids; map them to GSIS ids via rosters
pfr_to_gsis = rosters[['player_id', 'pfr_id']].dropna().drop_duplicates('pfr_id')
snaps = snap_counts.merge(pfr_to_gsis, left_on='pfr_player_id', right_on='pfr_id', how='inner')
snaps = snaps.groupby(['player_id', 'week'])['offense_pct'].max().reset_index()
snaps.columns = ['player_id', 'week', 'snap_share']

weekly_stats_df = weekly_stats_df.merge(snaps, on=['player_id', 'week'], how='left')
weekly_stats_df['snap_share'] = weekly_stats_df['snap_share'].fillna(0)

# SHIFTED 3-week and 5-week rolling averages for RBs only
weekly_stats_df.loc[rb_mask, 'rb_snap_share_3wk_avg'] = (
    weekly_stats_df[rb_mask].groupby('player_id')['snap_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[rb_mask, 'rb_snap_share_5wk_avg'] = (
    weekly_stats_df[rb_mask].groupby('player_id')['snap_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

rb_count = rb_mask.sum()
print(f"\n✓ Added RB-specific features for {rb_count} RB-week records:")
print("  - rb_opportunity_share_3wk_avg & 5wk_avg (shifted, rush attempts + targets / team total)")
print("  - rb_hvts_3wk_avg & 5wk_avg (shifted, high-value touches inside 10-yard line)")
print("  - rb_snap_share_3wk_avg & 5wk_avg (shifted, % of offensive snaps played)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")


Step 5: Engineering RB-Specific Features...

  Calculating Opportunity Share...
  Calculating High-Value Touches (yardline <= 10)...

✓ Added RB-specific features for 1423 RB-week records:
  - rb_opportunity_share_3wk_avg (shifted, rush attempts + targets / team total)
  - rb_hvts_3wk_avg (shifted, high-value touches inside 10-yard line)

Current shape: (5613, 39)


## CATEGORY 5: WR/TE-Specific Features

In [9]:
# ============================================================================
# CATEGORY 5: Wide Receiver (WR) / Tight End (TE) Specific Features
# ============================================================================
print("\nStep 6: Engineering WR/TE-Specific Features...\n")

# --- 5A: Target Share ---
print("  Calculating Target Share...")

# Team passing attempts per week (already calculated above in team_opportunities)
team_pass_attempts = pbp_2025.groupby(['posteam', 'week'])['pass_attempt'].sum().reset_index()
team_pass_attempts.columns = ['team', 'week', 'team_pass_attempts']

# Merge team passing attempts
weekly_stats_df = weekly_stats_df.merge(
    team_pass_attempts,
    left_on=['recent_team', 'week'],
    right_on=['team', 'week'],
    how='left'
)
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

# Calculate target share
weekly_stats_df['target_share'] = (
    weekly_stats_df['targets'] / weekly_stats_df['team_pass_attempts']
).fillna(0)

# SHIFTED rolling averages for WR/TE only
wr_te_mask = weekly_stats_df['position'].isin(['WR', 'TE'])

weekly_stats_df.loc[wr_te_mask, 'wr_te_target_share_3wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby('player_id')['target_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df.loc[wr_te_mask, 'wr_te_target_share_5wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby('player_id')['target_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 5B: Air Yards Share ---
print("  Calculating Air Yards Share...")

# Sum player air yards per week
player_air_yards = pbp_2025[
    pbp_2025['receiver_player_id'].notna() & 
    pbp_2025['air_yards'].notna()
].groupby(['receiver_player_id', 'week'])['air_yards'].sum().reset_index()
player_air_yards.columns = ['player_id', 'week', 'player_air_yards']

# Sum team air yards per week
team_air_yards = pbp_2025[
    pbp_2025['posteam'].notna() & 
    pbp_2025['air_yards'].notna()
].groupby(['posteam', 'week'])['air_yards'].sum().reset_index()
team_air_yards.columns = ['team', 'week', 'team_air_yards']

# Merge player air yards
weekly_stats_df = weekly_stats_df.merge(player_air_yards, on=['player_id', 'week'], how='left')
weekly_stats_df['player_air_yards'] = weekly_stats_df['player_air_yards'].fillna(0)

# Merge team air yards
weekly_stats_df = weekly_stats_df.merge(
    team_air_yards,
    left_on=['recent_team', 'week'],
    right_on=['team', 'week'],
    how='left'
)
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

# Calculate air yards share
weekly_stats_df['air_yards_share'] = (
    weekly_stats_df['player_air_yards'] / weekly_stats_df['team_air_yards']
).fillna(0)

# SHIFTED 3-week and 5-week rolling averages for WR/TE only
weekly_stats_df.loc[wr_te_mask, 'wr_te_air_yards_share_3wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby('player_id')['air_yards_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[wr_te_mask, 'wr_te_air_yards_share_5wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby('player_id')['air_yards_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 5C: WOPR (Weighted Opportunity Rating) ---
print("  Calculating WOPR...")

# Standard WOPR weights: 1.5 * target share + 0.7 * air yards share
weekly_stats_df['wopr'] = (
    1.5 * weekly_stats_df['target_share'] + 0.7 * weekly_stats_df['air_yards_share']
)

weekly_stats_df.loc[wr_te_mask, 'wr_te_wopr_3wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby('player_id')['wopr']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[wr_te_mask, 'wr_te_wopr_5wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby('player_id')['wopr']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 5D: Starting QB Historical Efficiency (Adjusted Yards per Attempt) ---
print("  Calculating Starting QB AY/A...")

# AY/A = (pass yards + 20*pass TDs - 45*INTs) / attempts, season-to-date
# through the PREVIOUS week (shifted to prevent leakage). Joined via the
# scheduled starting QB from the schedules data.
qb_hist = weekly_stats_df[weekly_stats_df['pass_attempts'] > 0][
    ['player_id', 'week', 'passing_yards', 'passing_tds', 'interceptions', 'pass_attempts']
].copy().sort_values(['player_id', 'week'])

g = qb_hist.groupby('player_id')
qb_hist['cum_yards'] = g['passing_yards'].transform(lambda x: x.shift(1).expanding().sum())
qb_hist['cum_tds'] = g['passing_tds'].transform(lambda x: x.shift(1).expanding().sum())
qb_hist['cum_ints'] = g['interceptions'].transform(lambda x: x.shift(1).expanding().sum())
qb_hist['cum_atts'] = g['pass_attempts'].transform(lambda x: x.shift(1).expanding().sum())

qb_hist['starting_qb_aya'] = (
    (qb_hist['cum_yards'] + 20 * qb_hist['cum_tds'] - 45 * qb_hist['cum_ints'])
    / qb_hist['cum_atts']
).replace([np.inf, -np.inf], 0).fillna(0)

qb_aya = qb_hist[['player_id', 'week', 'starting_qb_aya']].rename(columns={'player_id': 'starting_qb_id'})

weekly_stats_df = weekly_stats_df.merge(qb_aya, on=['starting_qb_id', 'week'], how='left')
weekly_stats_df['starting_qb_aya'] = weekly_stats_df['starting_qb_aya'].fillna(0)

wr_te_count = wr_te_mask.sum()
print(f"\n✓ Added WR/TE-specific features for {wr_te_count} WR/TE-week records:")
print("  - wr_te_target_share_3wk_avg & 5wk_avg (shifted, targets / team pass attempts)")
print("  - wr_te_air_yards_share_3wk_avg & 5wk_avg (shifted, air yards / team air yards)")
print("  - wr_te_wopr_3wk_avg & 5wk_avg (shifted, 1.5*target share + 0.7*air yards share)")
print("  - starting_qb_aya (shifted season-to-date AY/A of the team's starting QB)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")


Step 6: Engineering WR/TE-Specific Features...

  Calculating Target Share...
  Calculating Air Yards Share...

✓ Added WR/TE-specific features for 3526 WR/TE-week records:
  - wr_te_target_share_3wk_avg & 5wk_avg (shifted, targets / team pass attempts)
  - wr_te_air_yards_share_3wk_avg (shifted, air yards / team air yards)

Current shape: (5613, 47)


## CATEGORY 6: Matchup Features (Opponent Defense)

In [10]:
# ============================================================================
# CATEGORY 6: Matchup Features (Opponent Defense)
# ============================================================================
print("\nStep 7: Engineering Matchup Features (Opponent Defense Metrics)...\n")

# Get defensive team from play-by-play
# For each play, the defense is the team that does NOT have the ball (defteam)

# Calculate fantasy points allowed by defense per position per week
print("  Calculating fantasy points allowed by each defense...")

# Get the defense each player faced each week
# Defense = opponent team for that week (we already have 'opponent' from game_context)

# Aggregate fantasy points allowed by defense (opponent) to each position
def_points_allowed = weekly_stats_df.groupby(['opponent', 'week', 'position'])['fantasy_points_ppr'].sum().reset_index()
def_points_allowed.columns = ['defense', 'week', 'position', 'fp_allowed_this_week']

# Calculate season-to-date average fantasy points allowed by defense to each position
# This must be SHIFTED to prevent data leakage
def_points_allowed = def_points_allowed.sort_values(['defense', 'position', 'week'])

def_points_allowed['def_fp_allowed_cumsum'] = (
    def_points_allowed.groupby(['defense', 'position'])['fp_allowed_this_week']
    .transform(lambda x: x.shift(1).expanding().sum())
)

def_points_allowed['def_weeks_played'] = (
    def_points_allowed.groupby(['defense', 'position']).cumcount()
)

# Average = cumulative sum / weeks played (using shifted data)
def_points_allowed['opp_def_ppg_allowed'] = (
    def_points_allowed['def_fp_allowed_cumsum'] / def_points_allowed['def_weeks_played']
).fillna(0)

# Handle division by zero (first week has no history)
def_points_allowed['opp_def_ppg_allowed'] = def_points_allowed['opp_def_ppg_allowed'].replace([np.inf, -np.inf], 0)

# Merge defensive metrics into main dataframe
weekly_stats_df = weekly_stats_df.merge(
    def_points_allowed[['defense', 'week', 'position', 'opp_def_ppg_allowed']],
    left_on=['opponent', 'week', 'position'],
    right_on=['defense', 'week', 'position'],
    how='left'
)
weekly_stats_df.drop(columns=['defense'], inplace=True, errors='ignore')
weekly_stats_df['opp_def_ppg_allowed'] = weekly_stats_df['opp_def_ppg_allowed'].fillna(0)

# --- Opponent defensive efficiency (DVOA substitutes) ---
# DVOA is premium data; per CONTEXT rules we use yards-per-carry allowed and
# yards-per-play allowed instead. Both are SHIFTED season-to-date rates.
print("  Calculating opponent defensive efficiency (YPC / yards-per-play allowed)...")

def shifted_yards_rate(plays, rate_name):
    g = plays.groupby(['defteam', 'week']).agg(
        yards=('yards_gained', 'sum'),
        n_plays=('yards_gained', 'size')
    ).reset_index().sort_values('week')
    cum_yards = g.groupby('defteam')['yards'].transform(lambda x: x.shift(1).expanding().sum())
    cum_plays = g.groupby('defteam')['n_plays'].transform(lambda x: x.shift(1).expanding().sum())
    g[rate_name] = (cum_yards / cum_plays).fillna(0)
    return g[['defteam', 'week', rate_name]]

run_plays_def = pbp_2025[pbp_2025['play_type'] == 'run']
all_plays_def = pbp_2025[pbp_2025['play_type'].isin(['run', 'pass'])]

opp_ypc = shifted_yards_rate(run_plays_def, 'opp_def_ypc_allowed')
opp_ypp = shifted_yards_rate(all_plays_def, 'opp_def_ypp_allowed')

weekly_stats_df = weekly_stats_df.merge(
    opp_ypc, left_on=['opponent', 'week'], right_on=['defteam', 'week'], how='left'
).drop(columns=['defteam'], errors='ignore')

weekly_stats_df = weekly_stats_df.merge(
    opp_ypp, left_on=['opponent', 'week'], right_on=['defteam', 'week'], how='left'
).drop(columns=['defteam'], errors='ignore')

weekly_stats_df['opp_def_ypc_allowed'] = weekly_stats_df['opp_def_ypc_allowed'].fillna(0)
weekly_stats_df['opp_def_ypp_allowed'] = weekly_stats_df['opp_def_ypp_allowed'].fillna(0)

print("✓ Added matchup features:")
print("  - opp_def_ppg_allowed (shifted season-to-date avg FP allowed by opponent defense to this position)")
print("  - opp_def_ypc_allowed (shifted season-to-date yards per carry allowed, rush DVOA substitute)")
print("  - opp_def_ypp_allowed (shifted season-to-date yards per play allowed, DVOA substitute)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")


Step 7: Engineering Matchup Features (Opponent Defense Metrics)...

  Calculating fantasy points allowed by each defense...
✓ Added matchup features:
  - opp_def_ppg_allowed (shifted season-to-date avg FP allowed by opponent defense to this position)

Current shape: (5613, 48)


## CATEGORY 7: Supporting Cast Health (WR1/WR2)

In [ ]:
# ============================================================================
# CATEGORY 7: Supporting Cast Health (WR1/WR2 availability)
# ============================================================================
print("\nStep 8: Engineering Supporting Cast Health Features...\n")

# Identify each team's WR1/WR2 as the two WRs with the most cumulative targets
# through the PREVIOUS week (leakage-free), then check the injury report.
# Note: in week 1 there is no target history, so the ranking is arbitrary and
# the flag defaults to healthy unless those players are listed Out/Doubtful.
weeks = sorted(weekly_stats_df['week'].unique())
wr_rows = weekly_stats_df[weekly_stats_df['position'] == 'WR']

tgt_pivot = (
    wr_rows.pivot_table(index='player_id', columns='week', values='targets', aggfunc='sum')
    .reindex(columns=weeks)
    .fillna(0)
)
prior_targets = tgt_pivot.cumsum(axis=1).shift(1, axis=1).fillna(0)

prior_long = prior_targets.stack().reset_index()
prior_long.columns = ['player_id', 'week', 'prior_targets']
prior_long = prior_long.merge(player_team, on='player_id', how='left')

prior_long['wr_rank'] = (
    prior_long.groupby(['recent_team', 'week'])['prior_targets']
    .rank(method='first', ascending=False)
)
top2_wrs = prior_long[prior_long['wr_rank'] <= 2].copy()

# Players ruled Out or Doubtful on that week's injury report
inj_out = injuries[injuries['report_status'].isin(['Out', 'Doubtful'])][['gsis_id', 'week']].drop_duplicates()
inj_out['is_out'] = 1

top2_wrs = top2_wrs.merge(
    inj_out, left_on=['player_id', 'week'], right_on=['gsis_id', 'week'], how='left'
)
top2_wrs['is_out'] = top2_wrs['is_out'].fillna(0)

team_wr_health = top2_wrs.groupby(['recent_team', 'week'])['is_out'].sum().reset_index()
team_wr_health['wr1_wr2_healthy'] = (team_wr_health['is_out'] == 0).astype(int)

weekly_stats_df = weekly_stats_df.merge(
    team_wr_health[['recent_team', 'week', 'wr1_wr2_healthy']],
    on=['recent_team', 'week'],
    how='left'
)
weekly_stats_df['wr1_wr2_healthy'] = weekly_stats_df['wr1_wr2_healthy'].fillna(1).astype(int)

print("✓ Added supporting cast health features:")
print("  - wr1_wr2_healthy (1 = neither of the team's top-2 WRs is Out/Doubtful this week)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## Final Cleanup and Output Gold DataFrame

In [11]:
# ============================================================================
# FINAL CLEANUP: Fill NaNs and Prepare Gold DataFrame
# ============================================================================
print("\nStep 9: Final Cleanup...\n")

# List of all feature columns that might have NaNs
feature_cols = [
    'fantasy_points_3wk_avg', 'fantasy_points_5wk_avg',
    'qb_pass_attempts_3wk_avg', 'qb_pass_attempts_5wk_avg',
    'qb_rushing_yards_3wk_avg', 'qb_rushing_yards_5wk_avg',
    'rb_opportunity_share_3wk_avg', 'rb_opportunity_share_5wk_avg',
    'rb_hvts_3wk_avg', 'rb_hvts_5wk_avg',
    'rb_snap_share_3wk_avg', 'rb_snap_share_5wk_avg',
    'wr_te_target_share_3wk_avg', 'wr_te_target_share_5wk_avg',
    'wr_te_air_yards_share_3wk_avg', 'wr_te_air_yards_share_5wk_avg',
    'wr_te_wopr_3wk_avg', 'wr_te_wopr_5wk_avg',
    'starting_qb_aya',
    'opp_def_ppg_allowed', 'opp_def_ypc_allowed', 'opp_def_ypp_allowed',
    'opp_def_sack_rate', 'team_sack_rate_allowed',
    'implied_total', 'team_spread', 'temp', 'wind', 'rest_advantage',
    'is_dome', 'snap_share'
]

# Fill NaN values with 0
for col in feature_cols:
    if col in weekly_stats_df.columns:
        weekly_stats_df[col] = weekly_stats_df[col].fillna(0)

# Create the Gold DataFrame
gold_df = weekly_stats_df.copy()

print("="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)
print(f"\nGold DataFrame Shape: {gold_df.shape}")
print(f"Total Features: {len(gold_df.columns)}")
print(f"\nSample of engineered features:")

# Show a sample of key features
feature_sample_cols = [
    'week', 'player_name', 'position', 'recent_team', 'opponent',
    'fantasy_points_ppr', 'fantasy_points_3wk_avg',
    'implied_total', 'is_home', 'opp_def_ppg_allowed'
]
available_cols = [col for col in feature_sample_cols if col in gold_df.columns]
display(gold_df[available_cols].head(10))

print(f"\n✓ All NaN values in feature columns filled with 0")
print(f"✓ Gold DataFrame ready for XGBoost training!")
print(f"\nNext steps:")
print("  1. Split data into train/validation/test sets (by week or chronologically)")
print("  2. Define target variable (fantasy_points_ppr)")
print("  3. Train XGBoost model")
print("  4. Evaluate predictions")


Step 8: Final Cleanup...

FEATURE ENGINEERING COMPLETE

Gold DataFrame Shape: (5613, 48)
Total Features: 48

Sample of engineered features:


,week,player_name,position,recent_team,opponent,fantasy_points_ppr,fantasy_points_3wk_avg,implied_total,is_home,opp_def_ppg_allowed
0,15,P.Rivers,QB,IND,SEA,6.720000,0.000000,27.50,0.0,13.309231
1,16,P.Rivers,QB,IND,SF,16.160000,6.720000,25.50,1.0,17.424286
2,17,P.Rivers,QB,IND,JAX,7.400000,11.440000,26.00,1.0,17.272000
3,1,A.Rodgers,QB,PIT,NYJ,24.719999,0.000000,17.25,0.0,0.000000
4,2,A.Rodgers,QB,PIT,SEA,7.700000,24.719999,18.50,1.0,16.500000
5,3,A.Rodgers,QB,PIT,NE,11.860000,16.210000,21.50,0.0,17.150000
6,4,A.Rodgers,QB,PIT,MIN,12.280000,14.760000,22.00,1.0,12.020000
7,6,A.Rodgers,QB,PIT,CLE,17.500000,10.613333,16.00,1.0,16.336000
8,7,A.Rodgers,QB,PIT,CIN,22.559999,13.880000,20.00,0.0,19.440000
9,8,A.Rodgers,QB,PIT,GB,16.080000,17.446666,24.75,1.0,18.908000



✓ All NaN values in feature columns filled with 0
✓ Gold DataFrame ready for XGBoost training!

Next steps:
  1. Split data into train/validation/test sets (by week or chronologically)
  2. Define target variable (fantasy_points_ppr)
  3. Train XGBoost model
  4. Evaluate predictions


## Feature Summary and Data Quality Checks

In [12]:
# ============================================================================
# DATA QUALITY CHECKS AND FEATURE SUMMARY
# ============================================================================
print("\n" + "="*80)
print("FEATURE SUMMARY & DATA QUALITY REPORT")
print("="*80)

# 1. Overall Statistics
print(f"\n1. OVERALL STATISTICS:")
print(f"   Total Records: {len(gold_df):,}")
print(f"   Total Features: {len(gold_df.columns)}")
print(f"   Weeks Covered: {gold_df['week'].min()} - {gold_df['week'].max()}")
print(f"   Unique Players: {gold_df['player_id'].nunique():,}")

# 2. Position Breakdown
print(f"\n2. POSITION BREAKDOWN:")
print(gold_df['position'].value_counts())

# 3. Feature Categories
print(f"\n3. FEATURE CATEGORIES:")
print(f"   ✓ General Context: implied_total, team_spread, is_home, temp, wind, is_bad_weather, is_dome, rest_advantage")
print(f"   ✓ Rolling Averages: fantasy_points_3wk_avg, fantasy_points_5wk_avg")
print(f"   ✓ QB Features: qb_pass_attempts_3wk/5wk_avg, qb_rushing_yards_3wk/5wk_avg, opp_def_sack_rate, team_sack_rate_allowed")
print(f"   ✓ RB Features: rb_opportunity_share_3wk/5wk_avg, rb_hvts_3wk/5wk_avg, rb_snap_share_3wk/5wk_avg")
print(f"   ✓ WR/TE Features: wr_te_target_share_3wk/5wk_avg, wr_te_air_yards_share_3wk/5wk_avg, wr_te_wopr_3wk/5wk_avg, starting_qb_aya")
print(f"   ✓ Matchup Features: opp_def_ppg_allowed, opp_def_ypc_allowed, opp_def_ypp_allowed")
print(f"   ✓ Supporting Cast: wr1_wr2_healthy")

# 4. Missing Values Check
print(f"\n4. MISSING VALUES CHECK:")
missing = gold_df.isnull().sum()
if missing.sum() == 0:
    print("   ✓ No missing values in any column!")
else:
    print("   Columns with missing values:")
    print(missing[missing > 0])

# 5. Data Leakage Prevention Verification
print(f"\n5. DATA LEAKAGE PREVENTION VERIFICATION:")
print("   All rolling averages and historical metrics use .shift(1)")
print("   Week N predictions can only see data from Weeks 1 through N-1")
print("   ✓ Model is ready for time-series cross-validation")

# 6. Sample Feature Values for Top Performers
print(f"\n6. SAMPLE: Top 5 Fantasy Performances with Features")
top_performers = gold_df.nlargest(5, 'fantasy_points_ppr')[[
    'week', 'player_name', 'position', 'fantasy_points_ppr', 
    'fantasy_points_3wk_avg', 'implied_total', 'is_home', 'opp_def_ppg_allowed'
]]
display(top_performers)

print("\n" + "="*80)
print("Gold DataFrame is ready for ML model training!")
print("="*80)


FEATURE SUMMARY & DATA QUALITY REPORT

1. OVERALL STATISTICS:
   Total Records: 5,613
   Total Features: 48
   Weeks Covered: 1 - 22
   Unique Players: 608

2. POSITION BREAKDOWN:
position
WR    3526
RB    1423
QB     664
Name: count, dtype: int64

3. FEATURE CATEGORIES:
   ✓ General Context: implied_total, is_home, temp, wind, is_bad_weather, rest_advantage
   ✓ Rolling Averages: fantasy_points_3wk_avg, fantasy_points_5wk_avg
   ✓ QB Features: qb_pass_attempts_3wk/5wk_avg, qb_rushing_yards_3wk/5wk_avg
   ✓ RB Features: rb_opportunity_share_3wk_avg, rb_hvts_3wk_avg
   ✓ WR/TE Features: wr_te_target_share_3wk/5wk_avg, wr_te_air_yards_share_3wk_avg
   ✓ Matchup Features: opp_def_ppg_allowed

4. MISSING VALUES CHECK:
   Columns with missing values:
is_home                     5
is_bad_weather              5
opponent                    5
team_total_opportunities    5
team_pass_attempts          5
team_air_yards              5
dtype: int64

5. DATA LEAKAGE PREVENTION VERIFICATION:
   All r

,week,player_name,position,fantasy_points_ppr,fantasy_points_3wk_avg,implied_total,is_home,opp_def_ppg_allowed
3983,12,J.Gibbs,RB,55.400002,21.200000,18.25,1.0,26.069091
1704,10,J.Taylor,RB,49.599998,26.433334,21.00,1.0,23.324999
3967,16,P.Nacua,WR,46.500000,25.600000,22.00,0.0,41.907143
388,17,D.Henry,RB,45.599998,15.333333,20.50,0.0,19.093333
2348,15,K.Pitts,WR,45.599998,11.566667,24.75,0.0,44.776923



Gold DataFrame is ready for ML model training!
